# FUSION 2026 Stone Soup Figures

Refactored notebook with a deterministic pipeline and thin orchestration cells.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from fusion2026_st_pipeline import (
    build_config,
    build_platform,
    build_plugin_vs_ss_figure,
    build_simulator,
    build_target_truths,
    build_timesteps,
    build_tracker_figure,
    build_world_figure,
    generate_stonesoup_detections,
    print_summary,
    run_detection,
    run_tracking,
    write_figures,
)

/home/fin/miniconda3/envs/nereus-env/lib/python3.12/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


## 1) Configuration


In [3]:
cfg = build_config(seed=2000)
print(f"Total simulation duration: {cfg['total_duration_s']} seconds")
print(
    f"Number of timesteps: {cfg['sim']['num_steps']}, "
    f"Timestep interval: {cfg['sim']['time_interval'].total_seconds()} seconds"
)

Total simulation duration: 900.0 seconds
Number of timesteps: 180, Timestep interval: 5.0 seconds


## 2) Scenario Build + Simulation + Detection


In [4]:
platform = build_platform(cfg)
target_ground_truths, relative_bearing_ground_truths = build_target_truths(cfg, platform)
simulator = build_simulator(cfg, platform, target_ground_truths)

print("Running detection chain...")
all_detections, snr_map = run_detection(cfg, simulator)
timesteps = build_timesteps(cfg)

Running detection chain...


Generating Detections: 180it [00:23,  7.78it/s]


## 3) Tracking + Stone Soup Baseline Detections


In [5]:
relative_bearing_ground_truth = relative_bearing_ground_truths[0]
track = run_tracking(cfg["sim"]["start_time"], all_detections, relative_bearing_ground_truth)
stone_soup_detections = generate_stonesoup_detections(timesteps, relative_bearing_ground_truths)

## 4) Build Figures


In [6]:
world_fig = build_world_figure(timesteps, platform, target_ground_truths)
tracker_fig = build_tracker_figure(
    timesteps=timesteps,
    steering_azimuths_rad=cfg["beamforming"]["steering_azimuths_rad"],
    snr_map=snr_map,
    all_detections=all_detections,
    track=track,
    relative_bearing_ground_truth=relative_bearing_ground_truth,
)
plugin_vs_ss_fig = build_plugin_vs_ss_figure(timesteps, all_detections, stone_soup_detections)

world_fig.show()
tracker_fig.show()
plugin_vs_ss_fig.show()

## 5) Export + Summary


In [7]:
write_figures(
    {
        "st_world_picture.pdf": world_fig,
        "st_bf_tracker.pdf": tracker_fig,
        "st_plugin_vs_ss.pdf": plugin_vs_ss_fig,
    },
    output_dir="figs",
)
print_summary(cfg)


Random Scenario Summary:
Seed: 2000
Number of Targets: 1
Simulation Duration: 900.0 s across 180 timesteps
Timestep Interval: 5.0 s
Array: 50 sensors, 0.5 m spacing
Signal Sampling Rate: 500.0 Hz
Ambient Noise Level: 48.6 dB re 1 µPa
Steering Directions: 181
Array length: 25.0 m

Platform Initial State:
    Position: (-2000, 2000) m
    Velocity: (5.0, 0.0) m/s

Target 1:
    Frequencies: [ 90.77693112 118.10627705  36.91860048 127.29258553] Hz
    Amplitudes: [95.55775928 95.46793133 94.32662749 92.0471663 ] dB re 1 µPa
    Start Position: (0, 0) m
    Velocity: (0.0, 8.0) m/s
    Tonal Bandwidth: 0.99 Hz
    Noise Level: 78.9 dB re 1 µPa
